感知哈希 (Perceptual Hashing)
领域：图像检索、数字指纹、内容去重、相似图片检测。
通过提取图像的“指纹”（如颜色分布、DCT变换后的低频信息），比较两张图片在视觉上是否相似，常用于搜索引擎的“以图搜图”功能、盗版图片识别等。

感知哈希目标：把图片变成一串固定长度二进制“指纹”，内容相似的图，哈希也相似。
1. aHash（平均哈希，最简单）
步骤：
缩放到 8×8，转灰度，算全局平均灰度，每个像素：大于均值 → 1，否则 → 0，得到 64 位哈希
2. dHash（差值哈希，最常用）
步骤：
缩放到 9×8（多一列），转灰度，每行比较相邻像素：左 > 右 → 1，否则 → 0，得到 8×8 = 64 位哈希
 对亮度/对比度变化更鲁棒
3. pHash（真正的“感知哈希”）
步骤：
缩放到 32×32，转灰度，做 DCT（离散余弦变换），取左上角 8×8 低频系数，与中位数/均值比较 → 二值化，得到 64 位哈希
抗压缩、抗缩放、抗轻微滤镜

In [ ]:
# 使用Pillow
from importlib.util import find_spec

if find_spec('PIL') :
    pass
else:
    raise Exception("未安装pillow(python -m pip install pillow)")


In [7]:
from PIL import Image

image1=Image.open('data/威龙.jpg')
print(type(image1))

<class 'PIL.JpegImagePlugin.JpegImageFile'>


In [8]:
image1.show()

In [9]:
gray_image1 = image1.convert('L')

In [10]:
gray_image1.show()

In [11]:
# 使用Pillow
from importlib.util import find_spec

if find_spec('cv2') :
    pass
else:
    raise Exception("未安装cv(python -m pip install opencv-python)")

In [12]:
from PIL import Image, ImageEnhance, ImageFilter
import numpy as np

img = Image.open("data/威龙.jpg").convert("RGB")

# 1. 重压缩 (同图不同质)
img.save("data/q30.jpg", "JPEG", quality=30)

# 2. 缩放再放大 (分辨率欺骗)
w, h = img.size
img.resize((64, 64)).resize((w, h)).save("data/scale.jpg")

# 3. 裁剪头像 (局部相似，测试三视图头像与全身的关联)
# 原图左侧约1/3是头像，按此比例裁剪
crop_img = img.crop((0, 0, int(w*0.35), h))
# 为保持尺寸一致测试，可缩放回原尺寸或保留局部（建议保留局部看距离变化）
crop_img.save("data/crop_head.jpg")

# 4. 轻微旋转 (视角微变)
img.rotate(5, expand=True, fillcolor=(255,255,255)).save("data/rot5.jpg")

# 5. 无关图 (基准对照)
# 生成一个纯色/噪声图作为“无关图”示例
other = Image.fromarray(np.random.randint(0, 255, (h, w, 3), dtype=np.uint8))
other.save("data/other.jpg")

print("5张测试图生成完毕")

5张测试图生成完毕


In [14]:
import cv2
import numpy as np

def load_gray_cv(path, size):
    """加载图片并转为灰度、缩放"""
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f"无法读取图片: {path}")
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, size, interpolation=cv2.INTER_LANCZOS4)
    return resized.astype(np.float32)

# ---------- aHash（平均哈希）----------
def ahash(path, size=8):
    g = load_gray_cv(path, (size, size))
    mean = g.mean()
    bits = (g > mean).flatten().astype(np.uint8)
    return bits

# ---------- dHash（差值哈希）----------
def dhash(path, h=8, w=9):
    g = load_gray_cv(path, (w, h))
    diff = g[:, 1:] > g[:, :-1]
    return diff.flatten().astype(np.uint8)

# ---------- pHash（感知哈希，用OpenCV的DCT）----------
def phash(path, size=32, lowfreq=8):
    g = load_gray_cv(path, (size, size))
    
    # OpenCV的DCT变换
    dct = cv2.dct(g)
    
    # 取左上角低频区域
    low = dct[:lowfreq, :lowfreq]
    
    # 跳过DC分量（[0,0]位置），用中位数作为阈值
    med = np.median(low[1:, 1:])
    
    bits = (low > med).flatten().astype(np.uint8)
    return bits

# ---------- 汉明距离 ----------
def hamming(a, b):
    return int(np.sum(a != b))

# ---------- 哈希转十六进制字符串 ----------
def bits_to_hex(bits):
    bits_str = ''.join(bits.astype(str))
    return hex(int(bits_str, 2))[2:].zfill(len(bits)//4)


In [17]:
img1 = "data/q30.jpg"
img2 = "data/scale.jpg"

# aHash
h1 = ahash(img1)
h2 = ahash(img2)
print(f"aHash 汉明距离: {hamming(h1, h2)}")
print(f"aHash 指纹: {bits_to_hex(h1)}")

# dHash
h1 = dhash(img1)
h2 = dhash(img2)
print(f"dHash 汉明距离: {hamming(h1, h2)}")
print(f"dHash 指纹: {bits_to_hex(h1)}")


# pHash
h1 = phash(img1)
h2 = phash(img2)
print(f"pHash 汉明距离: {hamming(h1, h2)}")
print(f"pHash 指纹: {bits_to_hex(h1)}")

# 相似度百分比
similarity = 1 - hamming(h1, h2) / 64
print(f"pHash 相似度: {similarity:.2%}")

aHash 汉明距离: 3
aHash 指纹: ab2b6aafaaba1a1a
dHash 汉明距离: 11
dHash 指纹: 402018ab585a7010
pHash 汉明距离: 8
pHash 指纹: 9ff78ca7e89ce1f0
pHash 相似度: 87.50%
